# MedSegDiff — Asian Dataset Inference (Lambda)

Runs **inference only** using already-trained MedSegDiff checkpoints on the
`MRI_data_asian` thigh dataset.  No training is performed here.

## Where the models are kept (at least for me...)

| Location | Contents |
|---|---|
| `<<>>/dissector/medsegdiff/` | Package source: `dataset.py`, `unet.py`, `diffusion.py`, `predict.py` |
| `<<>>/dissector/eval_notebooks/medsegdiff_ckpts/` | **Trained checkpoints** — one `{muscle}_best.pt` and `{muscle}_latest.pt` per muscle |

**Important:** The checkpoint directory `medsegdiff_ckpts/` is not currently synced locally — the weights exist only on whichever Lambda instance ran `lambda_medsegdiff_.ipynb`. If that instance has been terminated you will need to re-run training first, or download the checkpoints before terminating the next instance.

Muscles with trained checkpoints: `R_gracilis`, `L_gracilis`, `R_sartorius`, `L_sartorius`

The model expects two input channels: **water** (channel 0) + **fat-fraction** (channel 1).
The Asian dataset has raw `Fat.nii.gz` rather than a pre-computed fat-fraction map, so
fat-fraction is computed on the fly as `fat / (water + fat + ε)`.

## Before running — upload to Lambda

```bash
# Trained checkpoints (required)
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff_ckpts/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_ckpts/

# MedSegDiff package
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/medsegdiff/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff/

# Asian MRI data
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/MRI_data_asian \
  ubuntu@<YOUR-LAMBDA-IP>:~/
```

## Download results when done

```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_asian_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff/asian_segs/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def _ensure(*pkgs):
    import importlib
    missing = [p for p in pkgs
               if importlib.util.find_spec(p.replace('-', '_')) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(missing))
    else:
        print('Already installed:', ', '.join(pkgs))

_ensure('SimpleITK', 'tqdm', 'torchvision')

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os, sys, glob
import numpy as np
import SimpleITK as sitk
import torch
import torch.nn.functional as F

MEDSEGDIFF_DIR = os.path.expanduser('~/medsegdiff')
CKPT_DIR       = os.path.expanduser('~/medsegdiff_ckpts')
DATA_ROOT      = os.path.expanduser('~/MRI_data_asian/MRI_data')
OUTPUT_DIR     = os.path.expanduser('~/medsegdiff_asian_segs')

for path, label in [
    (MEDSEGDIFF_DIR, 'medsegdiff package'),
    (CKPT_DIR,       'checkpoints'),
    (DATA_ROOT,      'Asian MRI data'),
]:
    ok = os.path.isdir(path)
    print(f'{"OK" if ok else "MISSING"}: {label} ({path})')
    if not ok:
        raise FileNotFoundError(f'Upload {label} first — see rsync commands above')

os.makedirs(OUTPUT_DIR, exist_ok=True)

if MEDSEGDIFF_DIR not in sys.path:
    sys.path.insert(0, MEDSEGDIFF_DIR)

from dataset   import GT_LABELS, _norm
from unet      import UNet
from diffusion import GaussianDiffusion

MUSCLES   = list(GT_LABELS)   # R_gracilis, L_gracilis, R_sartorius, L_sartorius
IMG_SIZE  = 256
BASE_CH   = 64
T_DIM     = 256
T_STEPS   = 1000
DDIM_STEPS = 50
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'\nMuscles : {MUSCLES}')
print(f'Device  : {DEVICE}')

In [ ]:
# ── Load checkpoints for all muscles ─────────────────────────────────────────
models = {}
diffusion = GaussianDiffusion(T=T_STEPS, device=DEVICE)

for muscle in MUSCLES:
    best_ckpt = os.path.join(CKPT_DIR, f'{muscle}_best.pt')
    if not os.path.exists(best_ckpt):
        print(f'[WARN] checkpoint not found for {muscle}: {best_ckpt}')
        continue

    ckpt       = torch.load(best_ckpt, map_location=DEVICE)
    saved_args = ckpt.get('args', {})
    # Model was trained with 2 channels (water + fat-fraction)
    img_ch     = saved_args.get('img_ch', 2)

    model = UNet(
        img_ch=img_ch,
        base=saved_args.get('base_ch', BASE_CH),
        t_dim=saved_args.get('t_dim', T_DIM),
    ).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()

    models[muscle] = (model, img_ch)
    print(f'  {muscle}: loaded (epoch {ckpt.get("epoch", "?")}, '
          f'best Dice {ckpt.get("best_dice", 0.0):.4f}, img_ch={img_ch})')

print(f'\nLoaded {len(models)}/{len(MUSCLES)} models.')

In [ ]:
# ── Discover Asian subjects ───────────────────────────────────────────────────
jobs = []
for subject in sorted(os.listdir(DATA_ROOT)):
    water_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'Water.nii.gz')
    fat_path   = os.path.join(DATA_ROOT, subject, 'Thigh', 'Fat.nii.gz')
    if not os.path.exists(water_path):
        continue
    has_fat = os.path.exists(fat_path)
    if not has_fat:
        print(f'  [WARN] {subject}: Fat.nii.gz not found — will use water-only (channel 2 = zeros)')
    jobs.append((subject, water_path, fat_path if has_fat else None))

print(f'Found {len(jobs)} subjects')
for subj, w, f in jobs[:5]:
    print(f'  {subj}  water=True  fat={f is not None}')

In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────

@torch.no_grad()
def segment_volume(model, img_ch, water_arr, fatfrac_arr):
    """Segment a 3D volume slice-by-slice. Returns binary uint8 (D, H, W)."""
    D, H, W = water_arr.shape
    pred_vol = np.zeros((D, H, W), dtype=np.uint8)
    model.eval()

    for sl in range(D):
        channels = [torch.from_numpy(_norm(water_arr[sl])).unsqueeze(0)]
        if img_ch == 2:
            ff_sl = fatfrac_arr[sl] if fatfrac_arr is not None else np.zeros_like(water_arr[sl])
            channels.append(torch.from_numpy(_norm(ff_sl)).unsqueeze(0))

        img_t = torch.cat(channels, dim=0) * 2.0 - 1.0          # (C, H, W) → [-1,1]
        img_r = F.interpolate(
            img_t.unsqueeze(0).to(DEVICE),
            size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False,
        )
        pred  = diffusion.ddim_sample(model, img_r, num_steps=DDIM_STEPS)
        pred_r = F.interpolate(pred, size=(H, W), mode='bilinear', align_corners=False)
        pred_vol[sl] = (pred_r.squeeze().cpu().numpy() > 0.0).astype(np.uint8)

    return pred_vol


for subject, water_path, fat_path in jobs:
    out_subdir = os.path.join(OUTPUT_DIR, subject, 'Thigh')
    out_npz    = os.path.join(out_subdir, 'Thigh_seg.npz')

    if os.path.exists(out_npz):
        # Check if all muscles are present
        existing_keys = set(np.load(out_npz).files)
        if set(models.keys()).issubset(existing_keys):
            print(f'Skipping (done): {subject}')
            continue

    print(f'\nProcessing: {subject}')
    os.makedirs(out_subdir, exist_ok=True)

    w_arr = sitk.GetArrayFromImage(sitk.ReadImage(water_path)).astype(np.float32)
    print(f'  Water shape: {w_arr.shape}')

    # Compute fat-fraction from Water + Fat images
    ff_arr = None
    if fat_path is not None:
        f_arr  = sitk.GetArrayFromImage(sitk.ReadImage(fat_path)).astype(np.float32)
        ff_arr = f_arr / (w_arr + f_arr + 1e-6)   # fat / (water + fat)

    # Load any previously written masks (allows resuming per-muscle)
    all_masks = {}
    if os.path.exists(out_npz):
        all_masks = dict(np.load(out_npz))

    for muscle, (model, img_ch) in models.items():
        if muscle in all_masks:
            print(f'  {muscle}: already done')
            continue
        print(f'  {muscle} ...', end=' ', flush=True)
        pred = segment_volume(model, img_ch, w_arr, ff_arr)
        all_masks[muscle] = pred
        voxels = int(pred.sum())
        print(f'{voxels:,} voxels')

    np.savez_compressed(out_npz, **all_masks)
    print(f'  Saved -> {out_npz}')

print('\nAll done.')

In [ ]:
# ── Sanity check ─────────────────────────────────────────────────────────────
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', 'Thigh', 'Thigh_seg.npz')))
print(f'Output NPZ files: {len(results)} / {len(jobs)}')
if results:
    sample = np.load(results[0])
    print(f'Sample: {results[0]}')
    for k in sample.files:
        arr = sample[k]
        print(f'  {k}: shape={arr.shape}  voxels={int(arr.sum()):,}')